# 듣고 질문에 답하기 유형 데이터 만들기

- 질문 생성 만들기
- 질문에 대한 오디오 만들기


## 질문 생성 Chain

In [ ]:
import json
from typing import List

from tqdm import tqdm
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser, CommaSeparatedListOutputParser
from pydantic import BaseModel, Field
# from langchain.schema import HumanMessage, AIMessage, StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import pandas as pd


import os


In [3]:
model = ChatOpenAI(model="gpt-5.6-luna")

### 질문 주제 샘플링하기

In [4]:
csv_parser = CommaSeparatedListOutputParser()

In [5]:
csv_format_instruction = csv_parser.get_format_instructions()

In [6]:
subjet_prompt_template = PromptTemplate.from_template(template="영어 시험에 나올 법한 일상적인 주제를 단어 형식으로 만들어줘.\n{format_instruction}",
                                                      partial_variables={"format_instruction": csv_format_instruction})

In [7]:
subject_chain = subjet_prompt_template | model | csv_parser

In [8]:
subject_chain.invoke({})

['family',
 'school',
 'friends',
 'hobbies',
 'sports',
 'food',
 'cooking',
 'shopping',
 'travel',
 'transportation',
 'weather',
 'seasons',
 'daily routines',
 'health',
 'exercise',
 'jobs',
 'technology',
 'environment',
 'holidays',
 'festivals',
 'music',
 'movies',
 'books',
 'pets',
 'home',
 'clothing',
 'restaurants',
 'plans',
 'invitations',
 'directions']

In [9]:
csv_parser.invoke(model.invoke(f"영어 시험에 나올 법한 일상적인 주제를 단어 형식으로 만들어줘.\n{csv_format_instruction}"))

['family',
 'school',
 'friends',
 'hobbies',
 'sports',
 'food',
 'health',
 'weather',
 'travel',
 'transportation',
 'shopping',
 'clothing',
 'daily routine',
 'housework',
 'technology',
 'jobs',
 'holidays',
 'plans',
 'places',
 'directions',
 'animals',
 'environment',
 'music',
 'movies',
 'books',
 'seasons',
 'time',
 'money',
 'communication',
 'future dreams']

In [10]:
subject_list = subject_chain.invoke({})

In [11]:
subject_list

['family',
 'school',
 'friends',
 'hobbies',
 'food',
 'shopping',
 'travel',
 'weather',
 'health',
 'exercise',
 'daily routine',
 'transportation',
 'holidays',
 'pets',
 'jobs',
 'technology',
 'environment',
 'music',
 'movies',
 'books',
 'sports',
 'plans',
 'feelings',
 'clothing',
 'housework',
 'restaurants',
 'directions',
 'communication',
 'culture',
 'social media']

In [12]:
subject_list = subject_list[:4]

In [13]:
subject_list

['family', 'school', 'friends', 'hobbies']

### 질문 만들기

In [14]:
model = ChatOpenAI(model="gpt-5.6-luna")

In [15]:
template = """\
# 이전 질문들 {prev_questions}
- 이전 질문들과는 다른 유형으로 만들어줘
- 영어 시험에 나올 법한 {input} 주제에 관한 쉬운 질문 하나 만들어줘.
- 상대방과 연관지어 만들어줘
- 한 문장만 만들어줘
- 여러 예시 만들지마
- 영어로"""

question_prompt_template = PromptTemplate.from_template(template=template)

In [16]:
question_chain = question_prompt_template | model | StrOutputParser()

In [19]:
question_list = []
for subject in tqdm(subject_list):
    question_list.append(question_chain.invoke({"input": subject, "prev_questions": question_list}))
    # question_list.append(question_chain.invoke({"input": subject}))

100%|██████████| 4/4 [00:09<00:00,  2.26s/it]


In [20]:
question_list

['Who are you closest to in your family?',
 'What subject do you enjoy most at school, and why?',
 'How do you usually spend time with your friends?',
 'What hobby would you recommend to me, and why?']

## 질문에 대한 오디오 파일 만들기

In [21]:
from openai import OpenAI

In [22]:
client = OpenAI()

In [23]:
def gen_speech_file(text, output_file_path):
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy", # alloy, echo, fable, onyx, nova, and shimmer
        input=text
    )
    response.stream_to_file(output_file_path)

In [24]:
!mkdir -p ./data/speaking__listen_and_answer

���� ������ �ùٸ��� �ʽ��ϴ�.


In [25]:
save_dir = "./data/speaking__listen_and_answer"

In [26]:
question_list

['Who are you closest to in your family?',
 'What subject do you enjoy most at school, and why?',
 'How do you usually spend time with your friends?',
 'What hobby would you recommend to me, and why?']

In [27]:
record_list = []

for i, q in tqdm(enumerate(question_list), total=len(question_list)):
    output_file_path = f"{save_dir}/question_{i}.wav"
    gen_speech_file(q, output_file_path)

    record = {"question": q, "audio_file_path": output_file_path}
    record_list.append(record)

C:\Users\jaeyo\AppData\Local\Temp\ipykernel_29468\455120697.py:7: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_file_path)
100%|██████████| 4/4 [00:12<00:00,  3.09s/it]


In [28]:
df = pd.DataFrame(record_list)
df

,question,audio_file_path
0,Who are you closest to in your family?,./data/speaking__listen_and_answer/question_0.wav
1,"What subject do you enjoy most at school, and ...",./data/speaking__listen_and_answer/question_1.wav
2,How do you usually spend time with your friends?,./data/speaking__listen_and_answer/question_2.wav
3,"What hobby would you recommend to me, and why?",./data/speaking__listen_and_answer/question_3.wav


In [29]:
df.to_csv(f"{save_dir}/question_and_audio.csv", index=False)

In [30]:
from IPython.display import Audio

In [31]:
Audio(f"{save_dir}/question_2.wav")